# 02. 모델 3개를 KFP run 3개로 비교하기

`mlflow/02-experiments.ipynb`의 모델 3개 비교를 KFP로 실행합니다. 파이프라인 정의는 하나이고, run마다 파라미터 `model_type`만 바꿉니다.
점수는 KFP Metrics 아티팩트로 남고, KFP 화면 **Experiments** → `income-classifier`의 **Compare runs**에서 비교합니다.

In [ ]:
from kfp import dsl

from common import load_data, prepare_data, register, submit, train_model, wait

MODEL_TYPES = ["decision-tree", "random-forest", "xgboost"]

In [ ]:
@dsl.pipeline(name="income-train", description="데이터 준비와 모델 학습. model_type으로 알고리즘을 고른다")
def income_train_pipeline(model_type: str = "xgboost"):
  """load-data, prepare-data, train-model을 순서대로 잇는다. 앞 두 단계는 입력이 같으면 KFP 캐시를 다시 쓴다."""
  data_task = load_data()
  split_task = prepare_data(data=data_task.outputs["data"])
  train_model(model_type=model_type, train=split_task.outputs["train"], test=split_task.outputs["test"])

파이프라인을 **Pipelines** 메뉴에 등록합니다. 처음 실행하면 파이프라인 `income-train`이 생기고, 다시 실행할 때마다 같은 파이프라인에 버전이 하나씩 쌓입니다. 노트북이 없는 동료도 화면에서 이 버전을 골라 **Create run**으로 같은 작업을 실행할 수 있습니다.

In [ ]:
train_ref = register(income_train_pipeline)

등록한 버전 하나로 `model_type`만 바꿔 run 3개를 만들고 모두 끝날 때까지 기다립니다. 3분쯤 걸립니다. 두 번째 run부터 `load-data`, `prepare-data`는 캐시를 씁니다.

In [ ]:
wait([submit(train_ref, f"train-{m}", {"model_type": m}) for m in MODEL_TYPES])